## DFT vs. MLIPs Energy Comparison Pipeline

This notebook compares 
$$E_{ads}^{MLIP} = E_{slab+ads}^{MLIP} - E_{slab}^{DFT} - \sum E_{refs}^{DFT} \\
\text{ and } \\ 
E_{ads}^{DFT} = E_{slab+ads}^{DFT} - E_{slab}^{DFT} - \sum E_{refs}^{DFT} $$
where refs are DFT potential energies of gas-phase reference molecules.

In [31]:
# TODO: write notebook description

# 12 plots instead of 4 (separate out models)
# try MAE by subtracting out different model/dft baselines and running MAE on that
# to consider: when doing E_adsorption, make sure E_total is on the same scale, or it might not work well?

#### 0. Configuration

In [32]:
from pathlib import Path
import numpy as np
from ase.io import iread, read, write

# wait for user input before running cells with expensive calculations
def confirm(prompt="Run long + expensive calculation?"):
    return input(prompt + " [y/n]: ").strip().lower() == 'y'

In [33]:
# builds a dict describing 1 DFT adsorbate run
# ads: single adsorbate string or two joined like ads1_ads2
# template: path template string with {ads}, {slab}, {config_id} placeholders;
#           define at call site (TEMPLATE in next cell) to handle
#           directory structure variants.

def make_run(ads, slab, config_id, template):
    return {
        "ads":       ads,
        "slab":      slab,
        "config_id": config_id,  # use arbitrary directory numbering to label ads config for now
        "path":      template.format(ads=ads, slab=slab, config_id=config_id),
    }

In [34]:
# USER INPUT HERE

# directory path template for adsorbate systems on YPdCu slab
YPDCU_TEMPLATE = "dft-data/YPdCu/{ads}_on_{slab}/{ads}_{slab}_{config_id}/vasprun.xml"

RUNS = (
    [make_run("CH3CHO",       "YO4PdCu", 3, YPDCU_TEMPLATE)] +
    [make_run("CO_CHO",       "YO4PdCu", i, YPDCU_TEMPLATE) for i in range(1, 15)] +
    [make_run("COCHO",        "YO4PdCu", i, YPDCU_TEMPLATE) for i in range(1, 19)] +
    [make_run("COOH",         "YO4PdCu", i, YPDCU_TEMPLATE) for i in range(1, 16)]
)

MODELS = [
        ("MACE",       "/Users/zschwab/miniconda3/envs/mlip-mace/bin/python",       "model-scripts/run_mace.py"),
        ("MatterSim",  "/Users/zschwab/miniconda3/envs/mlip-mattersim/bin/python",  "model-scripts/run_mattersim.py"),
        ("UMA",        "/Users/zschwab/miniconda3/envs/mlip-uma/bin/python",        "model-scripts/run_uma.py"),
]

#### 1. DFT data ingestion 
(extract all frames from DFT runs)

In [35]:
# load vasp trajectories into ASE Atoms obj.

# for trajectory files with multiple steps
def load_frames(path):
    frames = []
    for f in iread(path):
        frames.append(f)
    return frames

# for single-structure files (references, loads last by default)
def load_frame(path, index=-1):
    frame = read(path, index=index)
    return frame

In [36]:
# converts DFT trajectories -> .xyz files for MLIP runner scripts to read
# writes .xyz output files to hold model results later
confirm("Do you want to overwrite existing DFT trajectory files?")

for run in RUNS:
    write(f"dft-trajectories/{run['ads']}_{run['slab']}_{run['config_id']}_dft.xyz", load_frames(run['path']))

#### 2. run MLIPs (MACE, MatterSim, UMA)

In [38]:
confirm()

# runs each MLIP runner script as a subprocess using that model's own conda env
# each script reads the .xyz inputs and writes a .npz file (numpy filetype) 
# with predicted E, F(r)

import subprocess
import os

# new env inherits current environment as a starting point
env = os.environ.copy()

# disable GUI dependencies vscode auto-imports when launching a subprocess from
# a python notebook b/c they conflict with model dependencies and are unneeded
env["MPLBACKEND"] = "Agg"

for name, python, model_script in MODELS:
    for run in RUNS:
        run_id = f"{run['ads']}_{run['slab']}_{run['config_id']}"
        in_path = f"../mlip-dft-relaxed-comparison/dft-data/YPdCu/{run['ads']}_on_{run['slab']}/{run_id}/POSCAR"
        out_path = f"mlip-trajectories/{name}/{run_id}_mlip.traj"
        r = subprocess.run(
            [python, model_script, in_path, out_path],
            capture_output=True, text=True, env=env
        )
        
        if (r.returncode != 0):
            print(f"-- [{run_id}] ({name}): failed")
            if r.stdout: print(r.stdout)
            if r.stderr: print(r.stderr)
            print("Return code:", r.returncode)
        else:
            print(f"-- [{run_id}] ({name}): success")


-- [CH3CHO_YO4PdCu_3] (MACE): success
-- [CO_CHO_YO4PdCu_1] (MACE): success
-- [CO_CHO_YO4PdCu_2] (MACE): success
-- [CO_CHO_YO4PdCu_3] (MACE): success
-- [CO_CHO_YO4PdCu_4] (MACE): success
-- [CO_CHO_YO4PdCu_5] (MACE): success
-- [CO_CHO_YO4PdCu_6] (MACE): success
-- [CO_CHO_YO4PdCu_7] (MACE): success
-- [CO_CHO_YO4PdCu_8] (MACE): success
-- [CO_CHO_YO4PdCu_9] (MACE): success
-- [CO_CHO_YO4PdCu_10] (MACE): success
-- [CO_CHO_YO4PdCu_11] (MACE): success
-- [CO_CHO_YO4PdCu_12] (MACE): success
-- [CO_CHO_YO4PdCu_13] (MACE): success
-- [CO_CHO_YO4PdCu_14] (MACE): success
-- [COCHO_YO4PdCu_1] (MACE): success
-- [COCHO_YO4PdCu_2] (MACE): success
-- [COCHO_YO4PdCu_3] (MACE): success
-- [COCHO_YO4PdCu_4] (MACE): success
-- [COCHO_YO4PdCu_5] (MACE): success
-- [COCHO_YO4PdCu_6] (MACE): success
-- [COCHO_YO4PdCu_7] (MACE): success
-- [COCHO_YO4PdCu_8] (MACE): success
-- [COCHO_YO4PdCu_9] (MACE): success
-- [COCHO_YO4PdCu_10] (MACE): success
-- [COCHO_YO4PdCu_11] (MACE): success
-- [COCHO_YO4PdC

#### 3. Calculate adsorbate energies

In [39]:
# get bare slab & reference energies

REFS= {  # reference adsorbate energies
    "CO" :  None,
    "CO2":  None,
    "H2" :  None,
    "H2O":  None,
}

for mol in REFS:
    REFS[mol] = load_frame(f"../dft-ref-energy-data/adsorbates/{mol}_gas/vasprun.xml").get_potential_energy()

slab = load_frame("../dft-ref-energy-data/slabs/YO4PdCu/vasprun.xml")
slab_energy = slab.get_potential_energy()

In [ ]:
# define formulas for E_adsorbed = E_total - E_slab - E_adsorbates

def calc_CH3CHO(total_energy):
    return total_energy - slab_energy + REFS["H2O"] - 2*REFS["CO"] - 3*REFS["H2"]

def calc_CO_CHO(total_energy):
    CHO_energy_diff = REFS["H2O"] - REFS["CO2"] - (3/2)*REFS["H2"]
    return total_energy - slab_energy - REFS["CO"] - CHO_energy_diff

def calc_COCHO(total_energy):
    return total_energy - slab_energy - 2*REFS["CO"] - (1/2)*REFS["H2"]

def calc_COOH(total_energy):
    return total_energy - slab_energy - REFS["CO2"] - (1/2)*REFS["H2"]

In [44]:
# systematically rename folders

import os
import re

base = "../mlip-dft-relaxed-comparison/mlip-trajectories/MatterSim"

for name in os.listdir(base):
    # new_name = re.sub(r'(\d+)O(Y)', r'\2O\1', name)  # O4Y -> YO4
    # new_name = re.sub(r'100(?=_\d+$)', '', name)  # CO_CHO_YO4PdCu100_14 -> CO_CHO_YO4PdCu_14
    new_name = re.sub(r'_mlip\.traj$', '_mattersim.traj', name)
    if new_name != name:
        os.rename(
            os.path.join(base, name),
            os.path.join(base, new_name)
        )
        print(f"{name} -> {new_name}")

COCHO_YO4PdCu_14_mlip.traj -> COCHO_YO4PdCu_14_mattersim.traj
CO_CHO_YO4PdCu_12_mlip.traj -> CO_CHO_YO4PdCu_12_mattersim.traj
CO_CHO_YO4PdCu_5_mlip.traj -> CO_CHO_YO4PdCu_5_mattersim.traj
COOH_YO4PdCu_7_mlip.traj -> COOH_YO4PdCu_7_mattersim.traj
COOH_YO4PdCu_14_mlip.traj -> COOH_YO4PdCu_14_mattersim.traj
COCHO_YO4PdCu_7_mlip.traj -> COCHO_YO4PdCu_7_mattersim.traj
CO_CHO_YO4PdCu_8_mlip.traj -> CO_CHO_YO4PdCu_8_mattersim.traj
COOH_YO4PdCu_1_mlip.traj -> COOH_YO4PdCu_1_mattersim.traj
COCHO_YO4PdCu_12_mlip.traj -> COCHO_YO4PdCu_12_mattersim.traj
CO_CHO_YO4PdCu_14_mlip.traj -> CO_CHO_YO4PdCu_14_mattersim.traj
CO_CHO_YO4PdCu_3_mlip.traj -> CO_CHO_YO4PdCu_3_mattersim.traj
COCHO_YO4PdCu_1_mlip.traj -> COCHO_YO4PdCu_1_mattersim.traj
COOH_YO4PdCu_12_mlip.traj -> COOH_YO4PdCu_12_mattersim.traj
CO_CHO_YO4PdCu_13_mlip.traj -> CO_CHO_YO4PdCu_13_mattersim.traj
CO_CHO_YO4PdCu_4_mlip.traj -> CO_CHO_YO4PdCu_4_mattersim.traj
COCHO_YO4PdCu_15_mlip.traj -> COCHO_YO4PdCu_15_mattersim.traj
COOH_YO4PdCu_6_mli

In [ ]:
# calculate adsorbate energies from bare slab & reference energies
import re

ADSORBATE_CALCS = {
    "CH3CHO": calc_CH3CHO,
    "CO_CHO": calc_CO_CHO,
    "COCHO" : calc_COCHO,
    "COOH"  : calc_COOH,
}

pattern = re.compile(r"^(CH3CHO|CO_CHO|COCHO|COOH)_([^_]+)_(\d+)_(\w+)\.traj$")
e_ads = {model[0]: {} for model in MODELS}

for name, env, runner in MODELS:
    filepath = f"mlip-trajectories/{name}"
    for fname in os.listdir(filepath):
        m = pattern.match(fname)
        if m:
            ads_name = m.group(1)
            atoms = read(os.path.join(filepath, fname), index=-1)
            total_energy = atoms.get_potential_energy()
            config_id = fname.removesuffix(".traj")
            e_ads[name][config_id] = ADSORBATE_CALCS[ads_name](total_energy)

In [58]:
print(e_ads)
print(e_ads["UMA"])

{'MACE': {'COOH_YO4PdCu_6_mace': np.float64(5.245119989218757), 'CO_CHO_YO4PdCu_4_mace': np.float64(-32.635024873046866), 'CO_CHO_YO4PdCu_13_mace': np.float64(-33.787307587890616), 'COCHO_YO4PdCu_15_mace': np.float64(2.6385753164062566), 'CO_CHO_YO4PdCu_9_mace': np.float64(-33.755325166015616), 'COCHO_YO4PdCu_6_mace': np.float64(2.7361400136718816), 'COOH_YO4PdCu_15_mace': np.float64(4.955874383750007), 'COCHO_YO4PdCu_18_mace': np.float64(3.0817515859375066), 'CO_CHO_YO4PdCu_2_mace': np.float64(-33.91807541015624), 'COCHO_YO4PdCu_13_mace': np.float64(2.6438853750000066), 'COOH_YO4PdCu_13_mace': np.float64(4.785677850546882), 'COOH_YO4PdCu_7_mace': np.float64(4.967257440390632), 'COCHO_YO4PdCu_14_mace': np.float64(3.0199534902343816), 'CO_CHO_YO4PdCu_5_mace': np.float64(-33.52195724609374), 'CO_CHO_YO4PdCu_12_mace': np.float64(-34.12937912109374), 'COCHO_YO4PdCu_7_mace': np.float64(2.7851512441406316), 'COOH_YO4PdCu_14_mace': np.float64(4.961337030234382), 'CO_CHO_YO4PdCu_8_mace': np.fl

In [ ]:
# # TODO: get site type from initial ion coordinates in POSCAR

# from pymatgen.analysis.adsorption import AdsorbateSiteFinder
# from pymatgen.io.ase import AseAtomsAdaptor

In [ ]:
# # get initial adsorbate coordinates from POSCAR for each DFT run

# def get_ads_init_coords(folderpath: str):
#     poscar = read(folderpath + "/POSCAR")
#     print(poscar)

# get_ads_init_coords("dft-data/YPdCu/CO_CHO_on_YO4PdCu/CO_CHO_4OYPdCu100_1")

Atoms(symbols='C2HCu98O6PdY', pbc=True, cell=[12.515153631, 12.515153631, 25.30973], momenta=..., constraint=FixAtoms(indices=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49]))
